***leer los 256 bytes de header de cada canal***
- Cada header de canal en EDF sigue esta estructura (tamaño por campo en bytes, total 256):

<p align="center">
    <img src="../../assets/img/headerCanales.png" alt="Texto alternativo" width="450"/>
</p>

| Campo                            | Bytes (intervalo) | Longitud (bytes) | Comentarios                                                                 |
| -------------------------------- | ----------------- | ---------------- | --------------------------------------------------------------------------- |
| Label                            | 0–15              | 16               | Nombre del canal (ej. "EEG F3-REF")                                         |
| Transducer Type                  | 16–95             | 80               | Tipo de transductor o sensor                                                |
| Physical Dimension               | 96–103            | 8                | Unidad física (ej. "uV", "mV")                                              |
| Physical Minimum (float string)  | 104–111           | 8                | Valor mínimo físico correspondiente al mínimo digital                       |
| Physical Maximum (float string)  | 112–119           | 8                | Valor máximo físico correspondiente al máximo digital                       |
| Digital Minimum (int string)     | 120–127           | 8                | Mínimo valor digital (típicamente -32768)                                   |
| Digital Maximum (int string)     | 128–135           | 8                | Máximo valor digital (típicamente +32767)                                   |
| Prefiltering                     | 136–215           | 80               | Información sobre filtros aplicados antes del almacenamiento                |
| Number of samples in each record | 216–223           | 8                | Cuántas muestras hay por registro (usado para determinar duración y tamaño) |
| Reserved                         | 224–255           | 32               | Espacio reservado para uso futuro                                           |
| **Total**                        | 0–255             | 256              | Tamaño total del header por canal                                           |


In [ ]:
from typing import List, Dict
data_path = "../dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaauj/s004_2012/01_tcp_ar/aaaaaauj_s004_t000.edf"
# data_path = '../dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaedy/s001_2004/01_tcp_ar/aaaaaedy_s001_t002.edf'


In [10]:
def read_channel_headers_corrected(edf_path: str, n_channels: int = 30) -> List[Dict[str, str]]:
    with open(edf_path, 'rb') as f:
        f.seek(256)  # Saltar el header general
        header_bytes = f.read(n_channels * 256)

    # Definir los tamaños de cada campo (en bytes)
    field_sizes: Dict[str, int] = {
        "label": 16,
        "transducer_type": 80,
        "physical_dimension": 8,
        "physical_min": 8,
        "physical_max": 8,
        "digital_min": 8,
        "digital_max": 8,
        "prefiltering": 80,
        "samples_per_record": 8,
        "reserved": 32
    }

    
    offset: int = 0  # Inicializar offset para recorrer los campos del header
    fields_by_channel: Dict[str, List[str]] = {key: [] for key in field_sizes}
    # print("fields_by_channel:", fields_by_channel)

    # print("DEBUG INIT")
    # for key,value in field_sizes.items():
    #     print(f"Campo: {key}, Tamaño: {value} bytes")
    # print("DEBUG END")
    for field, size in field_sizes.items():
        total_size = size * n_channels
        field_data = header_bytes[offset:offset + total_size]
        for i in range(n_channels):
            start = i * size
            end = start + size
            fields_by_channel[field].append(field_data[start:end].decode('ascii', errors='ignore').strip())
        offset += total_size

    print("samples_per_recors")
    samples_per_record_int = [int(x) if x.isdigit() else x for x in fields_by_channel["samples_per_record"]]
    print(samples_per_record_int)

    # Combinar por canal
    channel_headers: List[Dict[str, str]] = []
    for i in range(n_channels):
        ch_header = {key: fields_by_channel[key][i] for key in field_sizes}
        channel_headers.append(ch_header)

    return channel_headers

# Ejemplo de uso
channel_headers = read_channel_headers_corrected(data_path, 30)

for i, ch in enumerate(channel_headers):
    print(f"Canal {i + 1}")
    for key, value in ch.items():
        print(f"\t{key}: {value}")
    print()


samples_per_recors
['', '', '', '', 'HP:-1.00', '0 Hz LP:', '-2.0 Hz', 'N:0.0', '', '', '', '', '', '', 'HP:-1.00', '0 Hz LP:', '-2.0 Hz', 'N:0.0', '', '', '', '', '', '', 'HP:-1.00', '0 Hz LP:', '-2.0 Hz', 'N:0.0', '', '']
Canal 1
	label: EEG FP1-REF
	transducer_type: EEG 31-REF      EEG 32-REF      EEG
	physical_dimension: 
	physical_min: uV
	physical_max: -29483.1
	digital_min: 29483.12
	digital_max: -32767
	prefiltering: -32767  -32767  32767   32767   32767   32767   32767   32767   32767   32767
	samples_per_record: 
	reserved: 

Canal 2
	label: EEG FP2-REF
	transducer_type: EEG
	physical_dimension: 
	physical_min: uV
	physical_max: -29483.1
	digital_min: 29483.12
	digital_max: -32767
	prefiltering: 32767   32767   32767   32767   32767   32767   32767   32767   32767   32767
	samples_per_record: 
	reserved: HP:-1.000 Hz LP:-2.0 Hz N:0.0

Canal 3
	label: EEG F3-REF
	transducer_type: EEG
	physical_dimension: 
	physical_min: uV
	physical_max: -29483.1
	digital_min: 29483.12
	digita